In [14]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from datetime import timedelta
from itertools import combinations

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.feature_selection import SequentialFeatureSelector

In [2]:



def build_model(model_type: str, use_early_stopping: bool = True):
    """
    Factory function to build a model by type.
    """
    model_type = model_type.lower()

    if model_type == "xgb":
        return xgb.XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=10,
            eval_metric="aucpr",  # PR-AUC for imbalanced data
            n_jobs=-1,
            random_state=42,
            early_stopping_rounds=50 if use_early_stopping else None,
        )

    elif model_type == "logreg":
        # Pipeline: scale features, then logistic regression
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="lbfgs",
                    n_jobs=-1,
                ))
            ]
        )

    elif model_type == "et":   # Extra Trees
        return ExtraTreesClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "rf":
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=10,
            n_jobs=-1,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "hgb":
        return HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=None,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.0,
            max_iter=300,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "svm":
        # Scale -> LinearSVC -> Calibrated for predict_proba
        base_svm = LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
        )
        calibrated_svm = CalibratedClassifierCV(
            base_svm,
            method="sigmoid",  # Platt scaling
            cv=3
        )

        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("svm", calibrated_svm),
            ]
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")


In [3]:
def build_feature_sets(df_train):
    """
    Build feature set definitions using 4 base groups:
    - static
    - recency
    - history
    - trend

    Then generate ALL non-empty combinations of these groups:
    (4 choose 1) + (4 choose 2) + (4 choose 3) + (4 choose 4) = 15.

    Returns
    -------
    feature_sets : dict
        {feature_set_name: [list_of_columns], ...}
    """
    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    all_features = [c for c in df_train.columns if c not in ignore_cols]

    # ---- 1. Build the 4 base groups ----

    # By prefix
    recency_prefix = "recent_"
    history_prefix = "history_"
    trend_prefix = "trend_"

    recency_features = [c for c in all_features if c.startswith(recency_prefix)]
    history_features = [c for c in all_features if c.startswith(history_prefix)]
    trend_features = [c for c in all_features if c.startswith(trend_prefix)]

    # Recency also includes this explicitly
    if "days_since_last_action" in all_features:
        recency_features.append("days_since_last_action")

    # Static = everything that is NOT recency/history/trend
    static_features = [
        c for c in all_features
        if c not in recency_features
        and c not in history_features
        and c not in trend_features
    ]

    base_groups = {
        "static": sorted(static_features),
        "recency": sorted(recency_features),
        "history": sorted(history_features),
        "trend": sorted(trend_features),
    }

    # ---- 2. Generate ALL non-empty combinations of these 4 groups ----

    feature_sets = {}

    group_names = list(base_groups.keys())

    for r in range(1, len(group_names) + 1):  # r = 1,2,3,4
        for combo in combinations(group_names, r):
            # combo is a tuple like ('static',) or ('static','recency') etc.
            name = "__".join(combo)  # e.g. "static", "static__recency", ...

            # union of all columns in the selected groups
            cols = set()
            for g in combo:
                cols.update(base_groups[g])

            cols = sorted(cols)

            # skip empty sets (paranoid check)
            if len(cols) == 0:
                continue

            feature_sets[name] = cols

    # Optional: add a friendly alias for everything
    feature_sets["all_features"] = sorted(all_features)

    return feature_sets


In [4]:
"""
def build_feature_sets(df_train):

    Build different feature set definitions, using naming patterns:
    - static: non-temporal features (no recent_/history_/trend_)
    - dynamic: temporal features (recent_/history_/trend_)

    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']

    all_features = [c for c in df_train.columns if c not in ignore_cols]

    dynamic_prefixes = ("recent_", "history_", "trend_")
    dynamic_features = [c for c in all_features if c.startswith(dynamic_prefixes)]
    static_features = [c for c in all_features if c not in dynamic_features]

    feature_sets = {
        "all_features": all_features,
        "static_only": static_features,
        "dynamic_only": dynamic_features,
        "static_plus_trend": static_features + [c for c in all_features if c.startswith("trend_")],
        # you can add more combos if you want
    }

    return feature_sets
"""

'\ndef build_feature_sets(df_train):\n\n    Build different feature set definitions, using naming patterns:\n    - static: non-temporal features (no recent_/history_/trend_)\n    - dynamic: temporal features (recent_/history_/trend_)\n\n    ignore_cols = [\'userId\', \'target_churn\', \'anchor_date\', \'ts_date\']\n\n    all_features = [c for c in df_train.columns if c not in ignore_cols]\n\n    dynamic_prefixes = ("recent_", "history_", "trend_")\n    dynamic_features = [c for c in all_features if c.startswith(dynamic_prefixes)]\n    static_features = [c for c in all_features if c not in dynamic_features]\n\n    feature_sets = {\n        "all_features": all_features,\n        "static_only": static_features,\n        "dynamic_only": dynamic_features,\n        "static_plus_trend": static_features + [c for c in all_features if c.startswith("trend_")],\n        # you can add more combos if you want\n    }\n\n    return feature_sets\n'

In [5]:
def evaluate_model_cv(X, y, groups, model_type: str, n_splits=5, threshold=0.5, plot_confusion=False):
    """
    Run GroupKFold CV for a given model + features, and return aggregated metrics.
    """
    gkf = GroupKFold(n_splits=n_splits)

    oof_pred_proba = np.zeros(len(y))
    oof_true = np.array(y)

    fold_roc = []
    fold_ap = []

    for i, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"  Fold {i + 1}/{n_splits} [{model_type}]")

        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]

        clf = build_model(model_type, use_early_stopping=True)

        if model_type == "xgb":
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        else:
            clf.fit(X_tr, y_tr)

        preds_proba = clf.predict_proba(X_val)[:, 1]
        oof_pred_proba[val_idx] = preds_proba

        fold_roc.append(roc_auc_score(y_val, preds_proba))
        fold_ap.append(average_precision_score(y_val, preds_proba))

    # Global metrics from OOF predictions
    roc = roc_auc_score(oof_true, oof_pred_proba)
    ap = average_precision_score(oof_true, oof_pred_proba)

    pred_labels = (oof_pred_proba >= threshold).astype(int)

    prec = precision_score(oof_true, pred_labels)
    rec = recall_score(oof_true, pred_labels)
    f1 = f1_score(oof_true, pred_labels)
    cm = confusion_matrix(oof_true, pred_labels)
    tn, fp, fn, tp = cm.ravel()

    print(f"  Mean ROC-AUC: {np.mean(fold_roc):.4f} | Global ROC-AUC: {roc:.4f}")
    print(f"  Mean AP (PR-AUC): {np.mean(fold_ap):.4f} | Global AP: {ap:.4f}")
    print(f"  Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print(f"  Confusion Matrix (OOF, thr={threshold}): TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    if plot_confusion:
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Pred: No Churn', 'Pred: Churn'],
            yticklabels=['True: No Churn', 'True: Churn']
        )
        plt.title(f'Confusion Matrix (OOF) - {model_type}')
        plt.ylabel('True label')
        plt.xlabel('Predicted label')
        plt.tight_layout()
        plt.show()

    metrics = {
        "roc_auc": roc,
        "pr_auc": ap,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

    return metrics


In [ ]:
def run_model_feature_search(df_train, model_types, thresholds =
                             [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9],
                             n_splits=5, results_csv_path=None):
    """
    Loop over (model_type, feature_set) combinations, evaluate via GroupKFold,
    and return a DataFrame of results.

    Parameters
    ----------
    df_train : pd.DataFrame with 'userId' and 'target_churn' columns
    model_types : list of strings, e.g. ["xgb", "logreg", "rf"]
    threshold : float, classification threshold for confusion-matrix metrics
    n_splits : int, number of GroupKFold splits
    results_csv_path : optional, if set, append/save results to this CSV
    """
    feature_sets = build_feature_sets(df_train)

    all_results = []
    for threshold in thresholds:
        for fs_name, fs_cols in feature_sets.items():
            print(f"\n=== Feature set: {fs_name} (n_features={len(fs_cols)}) ===")

            X = df_train[fs_cols]
            y = df_train['target_churn'].values
            groups = df_train['userId']

            for model_type in model_types:
                print(f"\n>> Evaluating model: {model_type} on feature set: {fs_name}")

                metrics = evaluate_model_cv(
                    X=X,
                    y=y,
                    groups=groups,
                    model_type=model_type,
                    n_splits=n_splits,
                    threshold=threshold,
                    plot_confusion=False,   # set True if you want plots each time
                )

                result_row = {
                    "model_type": model_type,
                    "feature_set": fs_name,
                    "n_features": len(fs_cols),
                    "threshold": threshold,
                }
                result_row.update(metrics)

                all_results.append(result_row)

                # Optional: save incremental progress so you don't lose results
                if results_csv_path is not None:
                    pd.DataFrame(all_results).to_csv(results_csv_path, index=False)

    results_df = pd.DataFrame(all_results)

    # Example: rank by PR-AUC first, then ROC-AUC
    results_df = results_df.sort_values(
        by=["pr_auc", "roc_auc"],
        ascending=False
    ).reset_index(drop=True)

    return results_df


In [7]:
# --- CONFIGURATION ---
# This is where you adjust the model's sensitivity
CONFIG = {
    'WINDOW_RECENT_DAYS': 7,    # Short-term period (recent trend)
    'WINDOW_HISTORY_DAYS': 14,  # Medium-term period (for comparison)
    'TARGET_WINDOW': 10,        # Prediction window (churn in the next X days)
    'STEP_SIZE': 7,             # We advance by 7 days with each iteration (Data Augmentation)
    'CHURN_EVENT': 'Cancellation Confirmation'
}

In [8]:
def load_and_clean_data(csv_path):
    """
    Loads the data and converts the temporal types.
    """
    print("Loading and cleaning the data...")
    df = pd.read_parquet(csv_path)

    df.drop("ts", axis=1)

    # On supprime les lignes sans userId valide (ex: utilisateurs non loggués)
    df = df[df['userId'].notna()]

    # Converting categorical columns to optimize memory
    categorical_cols = ['gender', 'level', 'page', 'method']
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    print(f"Data ready: {df.shape[0]} lines, from {df['time'].min()} to {df['time'].max()}")
    return df

In [9]:
df=load_and_clean_data("../../../data/churn-prediction-25-26/train.parquet")

Loading and cleaning the data...
Data ready: 17499636 lines, from 2018-10-01 00:00:01 to 2018-11-20 00:00:00


In [10]:
def device_type(s: str):
    if "Windows" in s:
        return "Windows"
    elif "Macintosh" in s:
        return "Macintosh"
    elif "Linux" in s:
        return "Linux"
    elif "iPad" in s:
        return "iPad"
    elif "iPhone" in s:
        return "iPhone"
    else:
        return "Different Device"


def exact_device_type(df: pd.DataFrame):
    """
    From a dataframe with at least ['userId', 'userAgent'],
    extract the device type and return a wide one-hot table by user.
    """
    df = df.copy()

    #Extract the part inside parentheses from the userAgent string
    df["device_used"] = df["userAgent"].apply(
        lambda x: x[x.index("(") : x.index(")") + 1]
        if isinstance(x, str) and "(" in x and ")" in x
        else None
    )
    # Map to simplified device categories
    df["exact_device"] = df["device_used"].apply(
        lambda x: device_type(x)
        if isinstance(x, str) else None)

    df_a = df[["userId", "exact_device"]]
    df_b = df_a.drop_duplicates().copy()

    df_b.loc[:, "val"] = 1

    # Pivot to get one column per device type
    df_c = df_b.pivot(index="userId", columns="exact_device", values="val")

    df_c = df_c.fillna(0.0)

    return df_c


def compute_features_at_anchor(df_window, anchor_date):
    """
    Compute features for all users who are active at a specific date (anchor_date).
    """
    # 1. Time slicing (Past only!)
    # "Recent" period: [Anchor - WINDOW_RECENT_DAYS, Anchor]
    start_recent = anchor_date - timedelta(days=CONFIG['WINDOW_RECENT_DAYS'])

    # "Historical" period: [Anchor - (WINDOW_HISTORY_DAYS + WINDOW_RECENT_DAYS), Anchor - WINDOW_RECENT_DAYS]
    # (we avoid overlap)
    start_history = start_recent - timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'])

    # Global filtering (keep only rows that are needed for feature computation)
    df_past = df_window[
        (df_window['time'] >= start_history) & (df_window['time'] <= anchor_date)
    ].copy()

    if df_past.empty:
        return None

    # Split into two subsets to compare trends
    mask_recent = df_past['time'] >= start_recent
    df_recent = df_past[mask_recent]
    df_history = df_past[~mask_recent]

    # --- A. STATIC FEATURES & STATE (Last known value) ---
    # Take the last row of each user to get their current state (Paid/Free, etc.)
    last_status = df_past.sort_values('time').groupby('userId').last()

    features = pd.DataFrame(index=df_past['userId'].unique())

    # Simple encoding: Level (1 if paid, 0 if free) - Only if the column exists
    if 'level' in last_status.columns:
        features['is_paid'] = (last_status['level'] == 'paid').astype(int)

    # Gender (1 if F, 0 if M for example)
    if 'gender' in last_status.columns:
        features['gender_F'] = (last_status['gender'] == 'F').astype(int)

    # Device type (one-hot by user) from userAgent if available
    if 'userAgent' in df_past.columns:
        # Use all past logs so we capture any device they used up to the anchor date
        device_features = exact_device_type(df_past[['userId', 'userAgent']])
        # Join on userId index
        features = features.join(device_features, how='left')

    # Recency: Days since the very last action
    last_action_date = df_past.groupby('userId')['time'].max()
    features['days_since_last_action'] = (
        (anchor_date - last_action_date).dt.total_seconds() / (3600 * 24)
    )

    # --- B. ACTIVITY FEATURES (Counts) ---
    # Helper function to avoid repetition
    def get_counts(df_sub, prefix):
        counts = df_sub.groupby('userId').agg({
            'page': 'count',      # Total actions
            'song': 'count',      # Total songs
            'sessionId': 'nunique',  # Total sessions
            'status': 'nunique'   # Number of distinct status codes (errors etc.)
        }).rename(columns={
            'page': f'{prefix}_actions',
            'song': f'{prefix}_songs',
            'sessionId': f'{prefix}_sessions',
            'status': f'{prefix}_status'
        })

        # Specific events (Thumbs Up, Thumbs Down, Errors, Add to Playlist, etc.)
        events = df_sub[df_sub['page'].isin([
            'Thumbs Up',
            'Thumbs Down',
            'Error',
            'Add to Playlist',
            'Add Friend',
            'Submit Upgrade',
            'Upgrade',
            'Downgrade',
            'Submit Downgrade',
            'Roll Advert'
        ])]

        if not events.empty:
            event_counts = pd.crosstab(events['userId'], events['page']).add_prefix(f'{prefix}_')
            counts = counts.merge(event_counts, on='userId', how='left')

        return counts

    feats_recent = get_counts(df_recent, 'recent')
    feats_history = get_counts(df_history, 'history')

    # Merge (including device features that were already joined above)
    features = features.join(feats_recent, how='left').join(feats_history, how='left').fillna(0)

    # --- C. TREND FEATURES (EVOLUTION) ---
    # This is key for churn detection: is activity decreasing?
    # We compute daily averages to compare periods of different lengths

    avg_songs_recent = features['recent_songs'] / CONFIG['WINDOW_RECENT_DAYS']
    avg_songs_history = features['history_songs'] / CONFIG['WINDOW_HISTORY_DAYS']

    # Trend ratio: (Recent + epsilon) / (History + epsilon)
    # If < 1: activity is going down
    epsilon = 0.1  # To avoid division by zero
    features['trend_song_consumption'] = (avg_songs_recent + epsilon) / (avg_songs_history + epsilon)

    # Error ratio (is the user facing more bugs recently?)
    if 'recent_Error' in features.columns and 'history_Error' in features.columns:
        features['trend_error_rate'] = (features['recent_Error'] + epsilon) / (features['history_Error'] + epsilon)

    # Status ratio (is the user facing more bugs recently?)
    if 'recent_status' in features.columns and 'history_status' in features.columns:
        features['trend_status_rate'] = (features['recent_status'] + epsilon) / (features['history_status'] + epsilon)

    # Thumbs Up ratio (is the user liking more songs?)
    if 'recent_Thumbs Up' in features.columns and 'history_Thumbs Up' in features.columns:
        features['trend_thumbs_up'] = (features['recent_Thumbs Up'] + epsilon) / (features['history_Thumbs Up'] + epsilon)

    # Thumbs Down ratio (is the user liking more songs?)
    if 'recent_Thumbs Down' in features.columns and 'history_Thumbs Down' in features.columns:
        features['trend_thumbs_down'] = (features['recent_Thumbs Down'] + epsilon) / (features['history_Thumbs Down'] + epsilon)

    # Add to Playlist ratio (is the user liking more songs?)
    if 'recent_Add to Playlist' in features.columns and 'history_Add to Playlist' in features.columns:
        features['trend_add_to_playlist'] = (features['recent_Add to Playlist'] + epsilon) / (features['history_Add to Playlist'] + epsilon)

    # Add Friend ratio (is the user liking more songs?)
    if 'recent_Add Friend' in features.columns and 'history_Add Friend' in features.columns:
        features['trend_add_friend'] = (features['recent_Add Friend'] + epsilon) / (features['history_Add Friend'] + epsilon)

    # Submit Upgrade ratio (is the user liking more songs?)
    if 'recent_Submit Upgrade' in features.columns and 'history_Submit Upgrade' in features.columns:
        features['trend_submit_upgrade'] = (features['recent_Submit Upgrade'] + epsilon) / (features['history_Submit Upgrade'] + epsilon)

    # Upgrade ratio (is the user liking more songs?)
    if 'recent_Upgrade' in features.columns and 'history_Upgrade' in features.columns:
        features['trend_upgrade'] = (features['recent_Upgrade'] + epsilon) / (features['history_Upgrade'] + epsilon)

    # Downgrade ratio (is the user liking more songs?)
    if 'recent_Downgrade' in features.columns and 'history_Downgrade' in features.columns:
        features['trend_downgrade'] = (features['recent_Downgrade'] + epsilon) / (features['history_Downgrade'] + epsilon)

    # Submit Downgrade ratio (is the user liking more songs?)
    if 'recent_Submit Downgrade' in features.columns and 'history_Submit Downgrade' in features.columns:
        features['trend_Submit downgrade'] = (features['recent_Submit Downgrade'] + epsilon) / (features['history_Submit Downgrade'] + epsilon)

    # Roll Advert ratio (is the user liking more songs?)
    if 'recent_Roll Advert' in features.columns and 'history_Roll Advert' in features.columns:
        features['trend_Roll Advert'] = (features['recent_Roll Advert'] + epsilon) / (features['history_Roll Advert'] + epsilon)

    # Add the anchor date for tracking
    features['anchor_date'] = anchor_date

    return features


In [11]:
def run_feature_engineering_pipeline(df_logs):
    """
    Runs the sliding window loop and generates the full Train/Test dataset.
    """
    print("Starting Sliding Window pipeline...")

    min_date = df_logs['time'].min()
    max_date = df_logs['time'].max()

    # We start only when we have enough history
    start_anchor = min_date + timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'] + CONFIG['WINDOW_RECENT_DAYS'])
    # We stop before the very end so we still have future data to build the target
    end_anchor = max_date - timedelta(days=CONFIG['TARGET_WINDOW'])

    current_date = start_anchor
    final_datasets = []

    while current_date <= end_anchor:
        print(f"Processing window for anchor date: {current_date.date()}")

        # 1. Generate features (X) for this anchor date
        # Use a wide view on the data to avoid unnecessary copies
        window_start = current_date - timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'] +
                                                     CONFIG['WINDOW_RECENT_DAYS'] + 1)
        df_view = df_logs[(df_logs['time'] >= window_start) & (df_logs['time'] <= max_date)]

        features = compute_features_at_anchor(df_view, current_date)

        if features is None or features.empty:
            current_date += timedelta(days=CONFIG['STEP_SIZE'])
            continue

        # 2. Generate the target (Y) - Future
        # Look into the future [Anchor, Anchor + TARGET_WINDOW days]
        future_mask = (df_view['time'] > current_date) & \
                      (df_view['time'] <= current_date + timedelta(days=CONFIG['TARGET_WINDOW']))

        future_data = df_view[future_mask]

        # Identify churners
        churn_users = future_data[future_data['page'] == CONFIG['CHURN_EVENT']]['userId'].unique()

        features['target_churn'] = 0
        features.loc[features.index.isin(churn_users), 'target_churn'] = 1

        final_datasets.append(features)

        # Move the sliding window forward
        current_date += timedelta(days=CONFIG['STEP_SIZE'])

    # Final concatenation
    if not final_datasets:
        print("Warning: No valid window generated (dataset too short?)")
        return pd.DataFrame()

    full_dataset = pd.concat(final_datasets).reset_index().rename(columns={'index': 'userId'})

    print(f"Pipeline finished. Dataset generated with shape: {full_dataset.shape}")
    return full_dataset


In [12]:
df_train_ready = run_feature_engineering_pipeline(df)

Starting Sliding Window pipeline...
Processing window for anchor date: 2018-10-22
Processing window for anchor date: 2018-10-29
Processing window for anchor date: 2018-11-05
Pipeline finished. Dataset generated with shape: (50548, 51)


In [ ]:
model_types = ["xgb", "hgb", "rf", "et", "logreg", "svm"]

results_df = run_model_feature_search(
    df_train=df_train_ready,
    model_types=model_types,
    n_splits=5,
    results_csv_path="model_feature_search_log_different_thresholds.csv",   # or None
)

print(results_df.head(10))



=== Feature set: static (n_features=7) ===

>> Evaluating model: xgb on feature set: static
  Fold 1/5 [xgb]
  Fold 2/5 [xgb]
  Fold 3/5 [xgb]
  Fold 4/5 [xgb]
  Fold 5/5 [xgb]


/Users/aaron/anaconda3/envs/pfds/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Mean ROC-AUC: 0.5703 | Global ROC-AUC: 0.5644
  Mean AP (PR-AUC): 0.0565 | Global AP: 0.0554
  Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
  Confusion Matrix (OOF, thr=0.5): TN=48185, FP=0, FN=2363, TP=0

>> Evaluating model: hgb on feature set: static
  Fold 1/5 [hgb]
  Fold 2/5 [hgb]
  Fold 3/5 [hgb]
  Fold 4/5 [hgb]
  Fold 5/5 [hgb]
  Mean ROC-AUC: 0.5677 | Global ROC-AUC: 0.5641
  Mean AP (PR-AUC): 0.0558 | Global AP: 0.0555
  Precision: 0.0575 | Recall: 0.6483 | F1: 0.1057
  Confusion Matrix (OOF, thr=0.5): TN=23094, FP=25091, FN=831, TP=1532

>> Evaluating model: rf on feature set: static
  Fold 1/5 [rf]
  Fold 2/5 [rf]
  Fold 3/5 [rf]
  Fold 4/5 [rf]
  Fold 5/5 [rf]
  Mean ROC-AUC: 0.5654 | Global ROC-AUC: 0.5630
  Mean AP (PR-AUC): 0.0550 | Global AP: 0.0548
  Precision: 0.0581 | Recall: 0.6416 | F1: 0.1066
  Confusion Matrix (OOF, thr=0.5): TN=23611, FP=24574, FN=847, TP=1516

>> Evaluating model: et on feature set: static
  Fold 1/5 [et]
  Fold 2/5 [et]
  Fold 3/5 [et]


/Users/aaron/anaconda3/envs/pfds/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Mean ROC-AUC: 0.5691 | Global ROC-AUC: 0.5673
  Mean AP (PR-AUC): 0.0557 | Global AP: 0.0557
  Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
  Confusion Matrix (OOF, thr=0.5): TN=48185, FP=0, FN=2363, TP=0

=== Feature set: recency (n_features=15) ===

>> Evaluating model: xgb on feature set: recency
  Fold 1/5 [xgb]
  Fold 2/5 [xgb]
  Fold 3/5 [xgb]
  Fold 4/5 [xgb]
  Fold 5/5 [xgb]
  Mean ROC-AUC: 0.6921 | Global ROC-AUC: 0.6914
  Mean AP (PR-AUC): 0.1178 | Global AP: 0.1136
  Precision: 0.1468 | Recall: 0.2124 | F1: 0.1736
  Confusion Matrix (OOF, thr=0.5): TN=45267, FP=2918, FN=1861, TP=502

>> Evaluating model: hgb on feature set: recency
  Fold 1/5 [hgb]
  Fold 2/5 [hgb]
  Fold 3/5 [hgb]
  Fold 4/5 [hgb]
  Fold 5/5 [hgb]
  Mean ROC-AUC: 0.6868 | Global ROC-AUC: 0.6868
  Mean AP (PR-AUC): 0.1118 | Global AP: 0.1085
  Precision: 0.0940 | Recall: 0.5391 | F1: 0.1600
  Confusion Matrix (OOF, thr=0.5): TN=35899, FP=12286, FN=1089, TP=1274

>> Evaluating model: rf on feature set: r

/Users/aaron/anaconda3/envs/pfds/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Fold 1/5 [xgb]
  Fold 2/5 [xgb]
  Fold 3/5 [xgb]
  Fold 4/5 [xgb]
  Fold 5/5 [xgb]
  Mean ROC-AUC: 0.6963 | Global ROC-AUC: 0.6961
  Mean AP (PR-AUC): 0.1169 | Global AP: 0.1130
  Precision: 0.1436 | Recall: 0.2298 | F1: 0.1768
  Confusion Matrix (OOF, thr=0.5): TN=44947, FP=3238, FN=1820, TP=543

>> Evaluating model: hgb on feature set: static__recency
  Fold 1/5 [hgb]
  Fold 2/5 [hgb]
  Fold 3/5 [hgb]
  Fold 4/5 [hgb]
  Fold 5/5 [hgb]
  Mean ROC-AUC: 0.6893 | Global ROC-AUC: 0.6893
  Mean AP (PR-AUC): 0.1112 | Global AP: 0.1086
  Precision: 0.0922 | Recall: 0.5493 | F1: 0.1579
  Confusion Matrix (OOF, thr=0.5): TN=35406, FP=12779, FN=1065, TP=1298

>> Evaluating model: rf on feature set: static__recency
  Fold 1/5 [rf]
  Fold 2/5 [rf]
  Fold 3/5 [rf]
  Fold 4/5 [rf]
  Fold 5/5 [rf]
  Mean ROC-AUC: 0.6642 | Global ROC-AUC: 0.6642
  Mean AP (PR-AUC): 0.0986 | Global AP: 0.0969
  Precision: 0.1548 | Recall: 0.1261 | F1: 0.1390
  Confusion Matrix (OOF, thr=0.5): TN=46558, FP=1627, FN=2

/Users/aaron/anaconda3/envs/pfds/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Mean ROC-AUC: 0.5908 | Global ROC-AUC: 0.5911
  Mean AP (PR-AUC): 0.0674 | Global AP: 0.0655
  Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
  Confusion Matrix (OOF, thr=0.5): TN=48185, FP=0, FN=2363, TP=0

=== Feature set: recency__history (n_features=29) ===

>> Evaluating model: xgb on feature set: recency__history
  Fold 1/5 [xgb]
  Fold 2/5 [xgb]
  Fold 3/5 [xgb]
  Fold 4/5 [xgb]
  Fold 5/5 [xgb]
  Mean ROC-AUC: 0.7260 | Global ROC-AUC: 0.7253
  Mean AP (PR-AUC): 0.1365 | Global AP: 0.1318
  Precision: 0.1585 | Recall: 0.2844 | F1: 0.2035
  Confusion Matrix (OOF, thr=0.5): TN=44616, FP=3569, FN=1691, TP=672

>> Evaluating model: hgb on feature set: recency__history
  Fold 1/5 [hgb]
  Fold 2/5 [hgb]
  Fold 3/5 [hgb]
  Fold 4/5 [hgb]
  Fold 5/5 [hgb]
  Mean ROC-AUC: 0.7207 | Global ROC-AUC: 0.7209
  Mean AP (PR-AUC): 0.1313 | Global AP: 0.1286
  Precision: 0.1014 | Recall: 0.5865 | F1: 0.1729
  Confusion Matrix (OOF, thr=0.5): TN=35902, FP=12283, FN=977, TP=1386

>> Evaluating m

In [18]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import GroupKFold

def forward_stepwise_selection(
    df_train,
    model_type="logreg",
    n_features_to_select=20,
    n_splits=5,
):
    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    all_features = [c for c in df_train.columns if c not in ignore_cols]

    X = df_train[all_features]
    y = df_train["target_churn"].values
    groups = df_train["userId"].values

    # Your existing model factory (logreg/xgb/rf/etc.)
    base_estimator = build_model(model_type, use_early_stopping=False)

    # --- KEY TRICK: precompute GroupKFold splits ---
    gkf = GroupKFold(n_splits=n_splits)
    cv_splits = list(gkf.split(X, y, groups))  # list of (train_idx, val_idx)

    sfs = SequentialFeatureSelector(
        estimator=base_estimator,
        n_features_to_select=n_features_to_select,
        direction="forward",             # forward stepwise
        scoring="average_precision",     # PR-AUC for imbalance
        cv=cv_splits,                    # use our precomputed splits
        n_jobs=-1,
    )

    # No groups argument here anymore
    sfs.fit(X, y)

    support = sfs.get_support()
    selected_features = [f for f, keep in zip(all_features, support) if keep]

    print(f"[Forward] Selected {len(selected_features)} features with {model_type}:")
    print(selected_features)

    return selected_features


In [ ]:
def generate_kaggle_submission(
    model,
    df_logs_full,
    feature_cols,
    all_test_user_ids,
    output_file='submission.csv',
    seuil=0.5
):
    """
    Generate a BINARY Kaggle submission, with robust handling of ID types
    and users without activity in the final window.
    """
    print("\n--- Generating ROBUST submission file (Binary Mode) ---")

    # 1. Reference date (anchor)
    last_date = df_logs_full['time'].max()
    print(f"Anchor date: {last_date}")

    # 2. Compute features for users active around the anchor date
    X_test_active = compute_features_at_anchor(df_logs_full, last_date)

    if X_test_active is None:
        X_test_active = pd.DataFrame()
        print("Warning: No active users in the last window!")

    # 3. Predictions for active users
    if not X_test_active.empty:
        # Ensure all expected feature columns are present
        for col in feature_cols:
            if col not in X_test_active.columns:
                X_test_active[col] = 0

        # Keep the same column order as in training
        X_test_active = X_test_active[feature_cols]

        active_probs = model.predict_proba(X_test_active)[:, 1]

        df_preds = pd.DataFrame({
            'userId': X_test_active.index,
            'prediction': active_probs
        })
    else:
        df_preds = pd.DataFrame(columns=['userId', 'prediction'])

    # 4. Merge with the full test user list
    # Start from all test user IDs provided (e.g., from sample_submission or test.csv)
    final_submission = pd.DataFrame({'userId': all_test_user_ids})

    # --- IMPORTANT: Harmonize ID types so the merge works correctly ---
    final_submission['userId'] = final_submission['userId'].astype(str)
    df_preds['userId'] = df_preds['userId'].astype(str)

    # Remove potential duplicates to avoid exploding the number of rows
    final_submission = final_submission.drop_duplicates(subset=['userId'])
    df_preds = df_preds.drop_duplicates(subset=['userId'])

    # Left join: keep all test users, attach predictions where available
    final_submission = final_submission.merge(df_preds, on='userId', how='left')

    # 5. Handle missing predictions (users inactive in the last window)
    avg_proba = final_submission['prediction'].mean()
    if pd.isna(avg_proba):
        avg_proba = 0.5  # fallback if no predictions at all

    missing_count = final_submission['prediction'].isna().sum()
    print(
        f"Inactive users: {missing_count} "
        f"(filled with mean probability: {avg_proba:.4f})"
    )

    final_submission['prediction'] = final_submission['prediction'].fillna(avg_proba)

    # 6. Apply threshold to get binary churn label
    final_submission['is_churn'] = (final_submission['prediction'] >= seuil).astype(int)

    # 7. Save submission file
    output_df = final_submission[['userId', 'is_churn']]
    output_df.columns = ['id', 'target']

    output_df.to_csv(output_file, index=False)
    print(f"Saved: {output_file} ({len(output_df)} rows)")

    return output_df


In [39]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

def evaluate_model_cv_simple(X, y, groups, model_type: str, n_splits=5):
    """
    Simple CV evaluator that returns global ROC-AUC & PR-AUC
    using GroupKFold and your build_model factory.
    """
    gkf = GroupKFold(n_splits=n_splits)
    oof_pred_proba = np.zeros(len(y))
    oof_true = np.array(y)

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]

        clf = build_model(model_type, use_early_stopping=False)
        clf.fit(X_tr, y_tr)

        preds_proba = clf.predict_proba(X_val)[:, 1]
        oof_pred_proba[val_idx] = preds_proba

    roc = roc_auc_score(oof_true, oof_pred_proba)
    ap = average_precision_score(oof_true, oof_pred_proba)

    return roc, ap


def forward_stepwise_selection_with_progress(
    df_train,
    model_type="logreg",
    n_features_to_select=20,
    n_splits=5,
):
    """
    Manual forward stepwise selection with progress logging.
    At each step, we add the feature that gives the highest PR-AUC
    (average_precision), and we log both PR-AUC & ROC-AUC.
    """
    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    all_features = [c for c in df_train.columns if c not in ignore_cols]

    X_full = df_train[all_features]
    y = df_train["target_churn"].values
    groups = df_train["userId"].values

    selected = []
    remaining = all_features.copy()
    history = []

    step = 0
    while remaining and len(selected) < n_features_to_select:
        step += 1
        best_feat = None
        best_pr_auc = -np.inf
        best_roc_auc = -np.inf

        print(f"\n=== Step {step} | currently selected: {len(selected)} features ===")

        for feat in remaining:
            current_feats = selected + [feat]
            X_sub = X_full[current_feats]

            roc, ap = evaluate_model_cv_simple(
                X=X_sub,
                y=y,
                groups=groups,
                model_type=model_type,
                n_splits=n_splits,
            )

            # choose by PR-AUC (ap)
            if ap > best_pr_auc:
                best_pr_auc = ap
                best_roc_auc = roc
                best_feat = feat

        # Update selected / remaining
        selected.append(best_feat)
        remaining.remove(best_feat)

        history.append({
            "step": step,
            "added_feature": best_feat,
            "n_features": len(selected),
            "roc_auc": best_roc_auc,
            "pr_auc": best_pr_auc,
        })

        print(
            f"--> Step {step}: added '{best_feat}' | "
            f"PR-AUC={best_pr_auc:.4f}, ROC-AUC={best_roc_auc:.4f}"
        )

    history_df = pd.DataFrame(history)

    print(f"\n[Forward] Final selected {len(selected)} features with {model_type}:")
    print(selected)

    return selected, history_df


In [26]:
sample = pd.read_csv('../../../data/churn-prediction-25-26/example_submission.csv')
all_users_list = sample['id'].unique()
all_users_list

array([1128274, 1782451, 1611542, ..., 1724758, 1879724, 1749215])

In [27]:
test_set=pd.read_parquet("../../../data/churn-prediction-25-26/test.parquet")

In [ ]:
best_feats_forward_logreg = forward_stepwise_selection(df_train=df_train_ready,
                           model_type="logreg")

[Forward] Selected 20 features with logreg:
['is_paid', 'gender_F', 'Linux', 'iPad', 'days_since_last_action', 'recent_actions', 'recent_Roll Advert', 'recent_Thumbs Up', 'recent_Upgrade', 'history_actions', 'history_Add to Playlist', 'history_Downgrade', 'history_Error', 'history_Roll Advert', 'history_Thumbs Down', 'history_Thumbs Up', 'history_Upgrade', 'trend_status_rate', 'trend_thumbs_down', 'trend_Submit downgrade']


In [36]:
best_feats_forward_xgb = forward_stepwise_selection(df_train=df_train_ready,
                           model_type="xgb")

[Forward] Selected 20 features with xgb:
['is_paid', 'days_since_last_action', 'recent_actions', 'recent_songs', 'recent_sessions', 'recent_Add to Playlist', 'recent_Submit Downgrade', 'recent_Thumbs Up', 'history_Add to Playlist', 'history_Downgrade', 'history_Roll Advert', 'history_Submit Upgrade', 'history_Thumbs Down', 'history_Thumbs Up', 'history_Upgrade', 'trend_error_rate', 'trend_thumbs_up', 'trend_thumbs_down', 'trend_downgrade', 'trend_Roll Advert']


In [ ]:
X_sel = df_train_ready[best_feats_forward_logreg]
y_sel = df_train_ready["target_churn"].values
groups = df_train_ready["userId"]

metrics_global = evaluate_model_cv(
    X=X_sel,
    y=y_sel,
    groups=groups,
    model_type="logreg",
    n_splits=5,
)

print("Global ROC-AUC:", metrics_global["roc_auc"])
print("Global PR-AUC:", metrics_global["pr_auc"])

  Fold 1/5 [logreg]
  Fold 2/5 [logreg]
  Fold 3/5 [logreg]
  Fold 4/5 [logreg]
  Fold 5/5 [logreg]
  Mean ROC-AUC: 0.7228 | Global ROC-AUC: 0.7231
  Mean AP (PR-AUC): 0.1376 | Global AP: 0.1347
  Precision: 0.0932 | Recall: 0.6471 | F1: 0.1630
  Confusion Matrix (OOF, thr=0.5): TN=33314, FP=14871, FN=834, TP=1529
Global ROC-AUC: 0.7230907678742587
Global PR-AUC: 0.13472547455443296


In [37]:
X_sel_xgb = df_train_ready[best_feats_forward_xgb]
y_sel_xgb = df_train_ready["target_churn"].values
groups = df_train_ready["userId"]

metrics_global = evaluate_model_cv(
    X=X_sel_xgb,
    y=y_sel_xgb,
    groups=groups,
    model_type="xgb",
    n_splits=5,
)

print("Global ROC-AUC:", metrics_global["roc_auc"])
print("Global PR-AUC:", metrics_global["pr_auc"])

  Fold 1/5 [xgb]
  Fold 2/5 [xgb]
  Fold 3/5 [xgb]
  Fold 4/5 [xgb]
  Fold 5/5 [xgb]
  Mean ROC-AUC: 0.7212 | Global ROC-AUC: 0.7206
  Mean AP (PR-AUC): 0.1377 | Global AP: 0.1341
  Precision: 0.1552 | Recall: 0.2844 | F1: 0.2008
  Confusion Matrix (OOF, thr=0.5): TN=44527, FP=3658, FN=1691, TP=672
Global ROC-AUC: 0.7205648757032195
Global PR-AUC: 0.1341167782380619


In [38]:
final_model = build_model("xgb", use_early_stopping=False)

final_model.fit(X_sel_xgb, y_sel_xgb)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'aucpr'


In [35]:
submit_df = generate_kaggle_submission(final_model, test_set, best_feats_forward_logreg, all_users_list, seuil=0.4)
submit_df["target"].mean()


--- Generating ROBUST submission file (Binary Mode) ---
Anchor date: 2018-11-20 00:00:00
Inactive users: 119 (filled with mean probability: 0.4866)
Saved: submission.csv (2904 rows)


np.float64(0.7000688705234159)